In [0]:
%run ./01_config

In [0]:
"""
12_backtest.py  —  Evaluation layer: temporal backtest (RQ4)

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 12 — Evaluation layer: temporal backtest (FR5 / RQ4, verdicts T4 and T5)

# Replays 2024–2025 under four policy regimes, using the generator's disclosed mechanics to
# simulate the counterfactual trajectories each policy would have produced on the identical
# fleet (Section 5.1). Using ground truth for the *simulation* is legitimate and is the
# point of the disclosed-truth design: counterfactuals require the true process. The
# *decisions* being evaluated — recommended intervals from notebook 10, cost-of-delay
# ordering from the notebook-08 model — were produced from transactional data alone.

# Regime  -  Intervals  -  Backlog ordering
# A baseline  -  as-is MPLA cycles  -  priority code
# B intervals only  -  recommended  -  priority code
# C ordering only  -  as-is  -  cost of delay
# D full framework  -  recommended  -  cost of delay

# B and C isolate each component's marginal contribution, so the headline delta is
# decomposable rather than confounded. Replications with common random numbers where the
# design permits; results reported as mean ± s.d. across replications.

# Honest limits of the replay, stated up front. Backlog response is modelled as a
# capacity-constrained daily queue, calibrated so the baseline regime reproduces the
# historical escalation rate — an approximation of the response-delay process that generated
# the data, not a reproduction of it. And the prioritizer's prior_failures feature is
# frozen at its cutoff value during the replay. Both belong in Section 5.4.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.linear_model import LogisticRegression

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

N_REPS = 20
COST_RATIO_BAND = [2.0, 3.0, 4.0, 5.0, 6.0]

OBS0 = datetime(2020, 1, 1)
CUTOFF_DAY = (pd.Timestamp(TRAIN_CUTOFF) - pd.Timestamp(OBS0)).days      # 1461
HORIZON = (pd.Timestamp(OBS_END) - pd.Timestamp(OBS0)).days              # 2191
SIM_DAYS = HORIZON - CUTOFF_DAY
print(f"replay window: day {CUTOFF_DAY+1}–{HORIZON} ({SIM_DAYS} days), {N_REPS} replications")

# Inputs: ground truth, fleet, recommendations, escalation model

for req in ["bronze_ground_truth_params", "results_interval_optimizer", "gold_escalation_training"]:
    if not spark.catalog.tableExists(tbl(req)):
        raise RuntimeError(f"{req} missing — run notebooks 03–11 first.")

gt = spark.table(tbl("bronze_ground_truth_params")).toPandas().set_index("EQTYP")
for c in ["weibull_beta", "weibull_eta_days", "pm_restoration_rho", "pm_cycle_days",
          "base_preventive_cost", "breakdown_cost_multiplier", "planned_repair_multiplier",
          "defect_rate_days", "escalation_base_hazard"]:
    gt[c] = gt[c].astype(float)

frailty = pd.read_csv(f"{GROUND_TRUTH}/equipment_frailty.csv").set_index("EQUNR")
eta_i_of = frailty["eta_effective_days"].to_dict()

eq = spark.sql(f"""
    SELECT e.equipment_id, e.equipment_class, e.iso14224_group, e.start_up_date,
           COALESCE(f.criticality, 3) AS criticality
    FROM {tbl('silver_equipment')} e
    LEFT JOIN {tbl('silver_functional_location')} f
           ON e.functional_location_id = f.functional_location_id
""").toPandas()
eq["install_day"] = (pd.to_datetime(eq.start_up_date) - pd.Timestamp(OBS0)).dt.days

opt = spark.table(tbl("results_interval_optimizer")).toPandas().set_index("equipment_class")
rec_cycle = {}
for cls in gt.index:
    if cls in opt.index and pd.notna(opt.loc[cls].get("T_recommended")):
        rec_cycle[cls] = float(opt.loc[cls, "T_recommended"])
    else:
        rec_cycle[cls] = float(gt.loc[cls, "pm_cycle_days"])   # run-to-failure flagged classes keep as-is
as_is_cycle = {cls: float(gt.loc[cls, "pm_cycle_days"]) for cls in gt.index}

# prior failures at cutoff, per item (frozen prioritizer feature)
pf = spark.sql(f"""
    SELECT equipment_id, COUNT(*) AS prior_failures
    FROM {tbl('silver_notification')}
    WHERE notification_type='M1' AND originating_notification_id IS NULL
      AND malfunction_start <= DATE'{TRAIN_CUTOFF}'
    GROUP BY equipment_id
""").toPandas().set_index("equipment_id")["prior_failures"].to_dict()

print(f"fleet {len(eq)} items | {len(gt)} classes | recommendations loaded for "
      f"{sum(1 for c in gt.index if rec_cycle[c] != as_is_cycle[c])} classes")

# Escalation model (retrained exactly as notebook 11, for scoring inside the replay)

esc_hist = spark.table(tbl("gold_escalation_training")).toPandas()
tr_hist = esc_hist[esc_hist.split == "train"]

def esc_design(df, cols=None):
    X = pd.get_dummies(df[["damage_code", "equipment_class"]], drop_first=True).astype(float)
    X["priority"] = df.priority.astype(float)
    X["criticality"] = df.criticality.fillna(2.5).astype(float)
    X["age_years"] = df.equipment_age_days.astype(float) / 365.25
    X["prior_failures"] = df.prior_failures.astype(float)
    if cols is not None:
        X = X.reindex(columns=cols, fill_value=0.0)
    return X

Xh = esc_design(tr_hist)
logit = LogisticRegression(max_iter=4000, C=1.0).fit(Xh, tr_hist.escalated.values)
MODEL_COLS = Xh.columns

premium = spark.sql(f"""
SELECT e.equipment_class,
       PERCENTILE_APPROX(CASE WHEN o.cost_type='BREAKDOWN' THEN o.total_actual_cost END,0.5)
         + AVG(CASE WHEN o.cost_type='BREAKDOWN' THEN o.downtime_valuation END)
         - PERCENTILE_APPROX(CASE WHEN o.cost_type='PLANNED_REPAIR' THEN o.total_actual_cost END,0.5)
         AS cost_premium
FROM {tbl('silver_order')} o JOIN {tbl('silver_equipment')} e ON o.equipment_id=e.equipment_id
GROUP BY e.equipment_class""").toPandas().set_index("equipment_class")["cost_premium"]
print("escalation model refit on train window; premiums loaded")

# Fleet state at the cutoff: virtual age and PM anchor per item, under true ρ

ev = spark.sql(f"""
    SELECT equipment_id, event_date, event_type FROM (
        SELECT equipment_id, malfunction_start AS event_date, 'FAILURE' AS event_type
        FROM {tbl('silver_notification')}
        WHERE notification_type='M1' AND originating_notification_id IS NULL
        UNION ALL
        SELECT equipment_id, basic_start, 'PM' FROM {tbl('silver_order')}
        WHERE order_class='PREVENTIVE'
    ) WHERE event_date <= DATE'{TRAIN_CUTOFF}'
""").toPandas()
ev["day"] = (pd.to_datetime(ev.event_date) - pd.Timestamp(OBS0)).dt.days
ev = ev.sort_values(["equipment_id", "day"])

state = {}
cls_of = dict(zip(eq.equipment_id, eq.equipment_class))
for r in eq.itertuples():
    state[r.equipment_id] = {"va": 0.0, "clock": float(r.install_day), "cls": r.equipment_class,
                             "crit": int(r.criticality), "group": r.iso14224_group,
                             "install": float(r.install_day)}
for eid, g in ev.groupby("equipment_id"):
    if eid not in state:
        continue
    s = state[eid]
    rho = float(gt.loc[s["cls"], "pm_restoration_rho"])
    for row in g.itertuples():
        gap = float(row.day) - s["clock"]
        if row.event_type == "FAILURE":
            s["va"] = 0.0
        else:
            s["va"] = (s["va"] + gap) * (1 - rho)
        s["clock"] = float(row.day)
print(f"state reconstructed for {len(state)} items "
      f"(mean virtual age at cutoff {np.mean([s['va'] for s in state.values()]):.0f} days)")

# Simulator

def residual_life(beta, eta, age, rng):
    u = rng.uniform(1e-4, 1 - 1e-4)
    age_term = (age / eta) ** beta if age > 0 else 0.0
    return max(eta * ((age_term - np.log(1 - u)) ** (1.0 / beta)) - age, 1.0)

def simulate_intrinsic(cycles, rep):
    """Weibull renewal with imperfect PM from the cutoff state. Returns per-event lists."""
    out = []   # (day, kind 'BD'|'PM', item, class, crit, u_cost)
    for eid, s0 in state.items():
        cls = s0["cls"]
        beta = float(gt.loc[cls, "weibull_beta"]); rho = float(gt.loc[cls, "pm_restoration_rho"])
        eta_i = float(eta_i_of.get(eid, gt.loc[cls, "weibull_eta_days"]))
        cyc = float(cycles[cls])
        rng = np.random.default_rng(abs(hash((rep, eid))) % (2**32))
        va, clock = s0["va"], max(s0["clock"], float(CUTOFF_DAY))
        nxt = s0["clock"] + cyc
        while nxt <= CUTOFF_DAY:
            nxt += cyc
        while clock < HORIZON:
            fd = clock + residual_life(beta, eta_i, va, rng)
            if fd <= nxt and fd < HORIZON:
                out.append((fd, "BD", eid, cls, s0["crit"], float(rng.uniform(0.85, 1.25))))
                va, clock = 0.0, fd; nxt = clock + cyc
            elif nxt < HORIZON:
                out.append((nxt, "PM", eid, cls, s0["crit"], float(rng.uniform(0.9, 1.1))))
                va = (va + (nxt - clock)) * (1 - rho)
                clock = nxt; nxt = clock + cyc
            else:
                break
    return out

def draw_defects(rep):
    """Common-random-number defect stream: identical across regimes within a replication."""
    defects = []
    for eid, s0 in state.items():
        cls = s0["cls"]
        p = gt.loc[cls]
        rng = np.random.default_rng(abs(hash((rep, eid, "defect"))) % (2**32))
        t = float(CUTOFF_DAY)
        while True:
            t += float(rng.exponential(p["defect_rate_days"]))
            if t >= HORIZON:
                break
            dmg = int(rng.integers(1, 9))
            prio = int(rng.choice([1, 2, 3, 4], p=[0.10, 0.25, 0.40, 0.25]))
            age_years = (t - s0["install"]) / 365.25
            lam = (p["escalation_base_hazard"] * (1 + 0.35 * (dmg >= 6))
                   * (1 + 0.15 * s0["crit"]) * (1 + 0.10 * age_years))
            defects.append({
                "arrival": t, "item": eid, "cls": cls, "crit": s0["crit"],
                "damage_code": f"DMG-{dmg}", "priority": prio, "age_years": age_years,
                "t_escalate": float(rng.exponential(1.0 / lam)),
                "u_bd": float(rng.uniform(0.85, 1.25)),
                "u_pr": float(rng.uniform(0.85, 1.15)),
            })
    d = pd.DataFrame(defects).sort_values("arrival").reset_index(drop=True)
    X = pd.DataFrame({
        "priority": d.priority.astype(float), "criticality": d.crit.astype(float),
        "age_years": d.age_years, "damage_code": d.damage_code,
        "equipment_class": d.cls,
        "prior_failures": d["item"].map(pf).fillna(0).astype(float),
    })
    Xd = esc_design(X.assign(equipment_age_days=X.age_years * 365.25,
                             escalated=0), MODEL_COLS)
    d["p_escalate"] = logit.predict_proba(Xd)[:, 1]
    d["cod_score"] = d.p_escalate * d.cls.map(premium).fillna(float(premium.median()))
    return d

def run_queue(defects, policy, capacity):
    """Daily capacity-limited service. Returns (escalated mask, service day array)."""
    n = len(defects)
    escalated = np.zeros(n, bool)
    arrival = defects.arrival.values
    esc_at = arrival + defects.t_escalate.values
    if policy == "priority":
        key = list(zip(defects.priority.values, arrival))
    else:
        key = list(zip(-defects.cod_score.values, arrival))
    queue, ai = [], 0
    order = np.argsort(arrival, kind="stable")
    for day in range(CUTOFF_DAY + 1, HORIZON + 1):
        while ai < n and arrival[order[ai]] <= day:
            queue.append(order[ai]); ai += 1
        # escalations strike before today's servicing
        still = []
        for j in queue:
            if esc_at[j] <= day:
                escalated[j] = True
            else:
                still.append(j)
        queue = still
        queue.sort(key=lambda j: key[j])
        for _ in range(capacity):
            if queue:
                queue.pop(0)
    for j in queue:                      # unserved at horizon: escalate if due, else drop
        if esc_at[j] <= HORIZON:
            escalated[j] = True
    return escalated

# Capacity calibration

# The queue's daily capacity is chosen so the baseline regime reproduces the historical
# escalation rate on the hold-out window. This anchors the replay to observed reality at the
# one point where it can be anchored; the calibrated value is reported, not hidden.

target = float(esc_hist[esc_hist.split == "test"].escalated.mean())
cal = {}
d0 = draw_defects(rep=0)
for C in range(1, 9):
    cal[C] = float(run_queue(d0, "priority", C).mean())
CAP = min(cal, key=lambda c: abs(cal[c] - target))
print("capacity -> baseline escalation rate:",
      {c: round(v, 3) for c, v in cal.items()})
print(f"historical hold-out escalation rate {target:.3f} -> calibrated capacity = {CAP}/day")

# Replay

REGIMES = {
    "A baseline": (as_is_cycle, "priority"),
    "B intervals only": (rec_cycle, "priority"),
    "C ordering only": (as_is_cycle, "cod"),
    "D full framework": (rec_cycle, "cod"),
}

rep_rows, daily = [], {r: np.zeros(SIM_DAYS + 1) for r in REGIMES}
group_tot = {r: {} for r in REGIMES}

for rep in range(N_REPS):
    defects = draw_defects(rep)
    intr_cache = {}
    for regime, (cycles, policy) in REGIMES.items():
        ckey = id(cycles)
        if ckey not in intr_cache:
            intr_cache[ckey] = simulate_intrinsic(cycles, rep)
        events = intr_cache[ckey]

        tot = dict(pm_cost=0.0, planned_cost=0.0,
                   bd_base_u=0.0, bd_base_u_crit=0.0,      # intrinsic, for recosting
                   esc_base_u=0.0, esc_base_u_crit=0.0,    # escalation, for recosting
                   n_pm=0, n_bd=0, n_esc=0, n_planned=0)
        day_cost = np.zeros(SIM_DAYS + 1)

        for (day, kind, eid, cls, crit, u) in events:
            base = float(gt.loc[cls, "base_preventive_cost"])
            di = min(int(day) - CUTOFF_DAY, SIM_DAYS)
            if kind == "PM":
                c = base * u
                tot["pm_cost"] += c; tot["n_pm"] += 1; day_cost[di] += c
            else:
                tot["bd_base_u"] += base * u
                tot["bd_base_u_crit"] += base * u * crit
                tot["n_bd"] += 1
                mult = float(gt.loc[cls, "breakdown_cost_multiplier"])
                day_cost[di] += base * u * mult * (1 + 0.25 * crit)

        escd = run_queue(defects, policy, CAP)
        for j, row in enumerate(defects.itertuples()):
            base = float(gt.loc[row.cls, "base_preventive_cost"])
            di = min(int(row.arrival) - CUTOFF_DAY, SIM_DAYS)
            if escd[j]:
                tot["esc_base_u"] += base * row.u_bd
                tot["esc_base_u_crit"] += base * row.u_bd * row.crit
                tot["n_esc"] += 1
                mult = float(gt.loc[row.cls, "breakdown_cost_multiplier"])
                day_cost[di] += base * row.u_bd * mult * (1 + 0.25 * row.crit)
            else:
                c = base * float(gt.loc[row.cls, "planned_repair_multiplier"]) * row.u_pr
                tot["planned_cost"] += c; tot["n_planned"] += 1; day_cost[di] += c

        m = 4.0   # calibrated ratio for headline totals
        tot["breakdown_cost"] = m * tot["bd_base_u"] + 0.25 * m * tot["bd_base_u_crit"]
        tot["escalation_cost"] = m * tot["esc_base_u"] + 0.25 * m * tot["esc_base_u_crit"]
        tot["total_cost"] = (tot["pm_cost"] + tot["planned_cost"]
                             + tot["breakdown_cost"] + tot["escalation_cost"])
        rep_rows.append({"rep": rep, "regime": regime, **tot})
        daily[regime] += np.cumsum(day_cost) / N_REPS

        # per-group totals for the transferability view (accumulate baseline & full only)
        if regime in ("A baseline", "D full framework"):
            for (day, kind, eid, cls, crit, u) in events:
                g = cls.split("-")[0]
                base = float(gt.loc[cls, "base_preventive_cost"])
                add = base * u if kind == "PM" else base * u * 4.0 * (1 + 0.25 * crit)
                group_tot[regime][g] = group_tot[regime].get(g, 0.0) + add / N_REPS
            for j, row in enumerate(defects.itertuples()):
                g = row.cls.split("-")[0]
                base = float(gt.loc[row.cls, "base_preventive_cost"])
                add = (base * row.u_bd * 4.0 * (1 + 0.25 * row.crit) if escd[j]
                       else base * float(gt.loc[row.cls, "planned_repair_multiplier"]) * row.u_pr)
                group_tot[regime][g] = group_tot[regime].get(g, 0.0) + add / N_REPS
    print(f"replication {rep + 1}/{N_REPS} complete")

res = pd.DataFrame(rep_rows)

# Results and pre-registered verdicts

agg = (res.groupby("regime")
          .agg(**{c: (c, "mean") for c in ["pm_cost", "planned_cost", "breakdown_cost",
                                           "escalation_cost", "total_cost",
                                           "n_pm", "n_bd", "n_esc"]},
               total_sd=("total_cost", "std"))
          .round(0).reindex(REGIMES))
display(spark.createDataFrame(agg.reset_index()))

A, B, C_, D = [agg.loc[r, "total_cost"] for r in REGIMES]
intrA = agg.loc["A baseline", "pm_cost"] + agg.loc["A baseline", "breakdown_cost"]
intrB = agg.loc["B intervals only", "pm_cost"] + agg.loc["B intervals only", "breakdown_cost"]
t4 = 100 * (intrA - intrB) / intrA
escA, escC = agg.loc["A baseline", "escalation_cost"], agg.loc["C ordering only", "escalation_cost"]
t5 = 100 * (escA - escC) / escA
rq4 = 100 * (A - D) / A

# replication-level uncertainty on the headline deltas
pv = res.pivot(index="rep", columns="regime", values="total_cost")
rq4_sd = float((100 * (pv["A baseline"] - pv["D full framework"]) / pv["A baseline"]).std())

verdicts = pd.DataFrame([
    {"test": "T4", "rq": "RQ2", "criterion": ">=15% cost-rate reduction vs calendar PM",
     "observed": f"{t4:.1f}% (PM + intrinsic breakdown cost, B vs A)",
     "status": "MET" if t4 >= 15 else "NOT MET"},
    {"test": "T5", "rq": "RQ3", "criterion": ">=20% cost-of-delay reduction vs priority/FIFO",
     "observed": f"{t5:.1f}% (escalation-attributable cost, C vs A)",
     "status": "MET" if t5 >= 20 else "NOT MET"},
    {"test": "T6", "rq": "RQ4", "criterion": "positive fleet-wide delta, robust across ratios/domains",
     "observed": f"{rq4:.1f}% ± {rq4_sd:.1f}% total-cost reduction (D vs A)",
     "status": "MET" if rq4 > 0 else "NOT MET"},
])
display(spark.createDataFrame(verdicts))
for r in verdicts.itertuples():
    print(f"[{r.status}] {r.test} ({r.rq})  {r.observed}")

# Figure B1 — cumulative cost trajectories under each regime

fig, ax = plt.subplots(figsize=(8.4, 4.6))
for (regime, series), col in zip(daily.items(), [MUTED, ACCENT, GREEN, WARM]):
    x = pd.date_range(pd.Timestamp(TRAIN_CUTOFF) + pd.Timedelta(days=1), periods=len(series))
    ax.plot(x, series, lw=1.7, color=col, label=regime)
ax.set_ylabel("cumulative maintenance cost (mean of replications)")
ax.legend(frameon=False, loc="upper left")
emit(fig, "b1_cumulative_cost")

# Figure B2 — sensitivity of the framework delta to the breakdown cost ratio

sens_rows = []
for m in COST_RATIO_BAND:
    tm = {}
    for regime in REGIMES:
        rr = res[res.regime == regime]
        tm[regime] = float((rr.pm_cost + rr.planned_cost
                            + m * rr.bd_base_u + 0.25 * m * rr.bd_base_u_crit
                            + m * rr.esc_base_u + 0.25 * m * rr.esc_base_u_crit).mean())
    sens_rows.append({"cost_ratio": m,
                      "full_framework_saving_pct":
                          round(100 * (tm["A baseline"] - tm["D full framework"]) / tm["A baseline"], 1)})
sens = pd.DataFrame(sens_rows)
display(spark.createDataFrame(sens))

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(sens.cost_ratio, sens.full_framework_saving_pct, color=WARM, lw=1.8, marker="o")
ax.axhline(0, color=MUTED, lw=.9)
ax.axvline(4.0, color=GRID, lw=1, ls="--")
ax.set_xlabel("assumed breakdown : preventive cost ratio")
ax.set_ylabel("full-framework total-cost saving vs baseline (%)")
ax.set_title("dashed = calibrated 4.0x; the zero crossing (if any) is the break-even boundary",
             fontsize=8, color=MUTED, loc="left", pad=8)
emit(fig, "b2_cost_ratio_sensitivity")

# Figure B3 — transferability: framework benefit by ISO 14224 top-level group

groups = sorted(group_tot["A baseline"])
sav = [100 * (group_tot["A baseline"][g] - group_tot["D full framework"].get(g, 0.0))
       / group_tot["A baseline"][g] for g in groups]
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.bar(groups, sav, color=ACCENT, width=.6)
ax.axhline(0, color=MUTED, lw=.9)
ax.set_ylabel("total-cost saving, full framework vs baseline (%)")
ax.set_title("same code, same models, no re-tuning per domain", fontsize=8, color=MUTED,
             loc="left", pad=8)
emit(fig, "b3_transferability")
transf = pd.DataFrame({"iso14224_group": groups, "saving_pct": np.round(sav, 1)})
display(spark.createDataFrame(transf))

# Persist

from pyspark.sql.functions import lit

for name, df in [("results_backtest", agg.reset_index()),
                 ("results_backtest_sensitivity", sens),
                 ("results_backtest_transferability", transf),
                 ("results_rq234_verdict", verdicts)]:
    (spark.createDataFrame(df).withColumn("dataset_version", lit(DATASET_VERSION))
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl(name)))
    df.to_csv(f"{EXPORTS}/{name}.csv", index=False)
    print(f"{name} written + exported")

print(f"\ncalibrated queue capacity: {CAP} defects/day (report in Section 5.1)")
print("Fill Section 5.2, the Section 5.5 verdict rows, Section 5.1 and the abstract from "
      "results_rq234_verdict and results_backtest; mirror the same numbers everywhere.")